In [9]:
import pandas as pd
import re
from datetime import datetime

def parse_adults(text):
    patterns = [
        (r'(\d+)\s*взрослых?', True),
        (r'(\d+)\s*взр', True),
        (r'двое\s*взрослых', False),
        (r'два\s*взрослых', False),
        (r'2\s*взр', False),
    ]
    for pat, has_group in patterns:
        match = re.search(pat, text.lower())
        if match:
            return int(match.group(1)) if has_group else 2
    return 2

def parse_children(text):
    patterns = [
        (r'(\d+)\s*дет[еяй]', True),
        (r'(\d+)\s*ребёнка?', True),
        (r'ребёнок\s*(\d+)\s*лет', True),
    ]
    for pat, has_group in patterns:
        match = re.search(pat, text.lower())
        if match and has_group:
            return int(match.group(1))
    match = re.search(r'\(([\d,]+)\)', text)
    if match:
        return len(re.findall(r'\d+', match.group(1)))
    return 0

def parse_dates_and_nights(text):
    date_range = re.search(r'(\d{1,2}\.\d{1,2})(?:-|–|—|\.\.| по )(\d{1,2}\.\d{1,2})', text)
    if date_range:
        start, end = date_range.group(1), date_range.group(2)
        try:
            start_dt = datetime.strptime(start + ".2024", "%d.%m.%Y")
            end_dt = datetime.strptime(end + ".2024", "%d.%m.%Y")
            if end_dt < start_dt:
                end_dt = end_dt.replace(year=2025)
            nights = (end_dt - start_dt).days
            return start_dt.strftime("%Y-%m-%d"), nights
        except:
            pass
    days_match = re.search(r'на\s*(\d+)\s*(дней|дня|суток|недель|недели)', text.lower())
    if days_match:
        days = int(days_match.group(1))
        if 'недел' in days_match.group(2):
            days *= 7
        return None, days
    return None, None

def parse_price(text):
    price_match = re.search(r'(\d+)\s*[-–]\s*(\d+)\s*[₽руб]', text)
    if price_match:
        return int(price_match.group(2))
    price_match = re.search(r'до\s*(\d+)\s*[₽руб]', text.lower())
    if price_match:
        return int(price_match.group(1))
    price_match = re.search(r'(\d+)\s*[₽руб]', text)
    if price_match:
        return int(price_match.group(1))
    return None

def parse_remarks(text):
    keywords = ['недалеко от моря', 'рядом с морем', 'у моря', 'с кухней', 'со всеми удобствами',
                'су в номере', 'недорого', 'эконом', 'стандарт', 'оборудованным пляжем',
                'развлекательная инфраструктура', 'в шаговой доступности', 'тайные тропы']
    remarks = [kw for kw in keywords if kw.lower() in text.lower()]
    return ', '.join(remarks) if remarks else None

df = pd.read_csv('rental_26.csv', sep=';')
df['count_adults'] = df['text'].apply(parse_adults)
df['count_children'] = df['text'].apply(parse_children)

date_nights = df['text'].apply(parse_dates_and_nights)
df['start_date'] = date_nights.apply(lambda x: x[0])
df['nights'] = date_nights.apply(lambda x: x[1])

df['price_per_day'] = df['text'].apply(parse_price)
df['remarks'] = df['text'].apply(parse_remarks)

df_annotated = df[['text', 'count_adults', 'count_children', 'start_date', 'nights', 'price_per_day', 'remarks']]
df_annotated.to_csv('rental_26_annotated.csv', sep=';', index=False, encoding='utf-8-sig')
print("Файл rental_26_annotated.csv успешно создан!")

Файл rental_26_annotated.csv успешно создан!


In [10]:
import os
import pandas as pd
import json
from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

load_dotenv()
API_KEY = os.getenv("GIGA_KEY")
if not API_KEY:
    raise ValueError("Переменная окружения GIGA_KEY не найдена. Проверьте файл .env и перезапустите ячейку.")

llm = GigaChat(
    credentials=API_KEY,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.1,
    max_tokens=2000,
    response_format={"type": "json_object"}   # важно!
)

json_schema = {
    "type": "object",
    "properties": {
        "count_adults": {"type": "integer"},
        "count_children": {"type": "integer"},
        "start_date": {"type": ["string", "null"]},
        "nights": {"type": ["integer", "null"]},
        "price_per_day": {"type": ["integer", "null"]},
        "remarks": {"type": ["string", "null"]}
    },
    "required": ["count_adults", "count_children"]
}

parser = JsonOutputParser(schema=json_schema)

detailed_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Ты — эксперт по анализу текстов заявок на аренду жилья. Верни **только JSON** без комментариев и пояснений.

Имена полей (строго): count_adults, count_children, start_date, nights, price_per_day, remarks.
- count_adults: число взрослых.
- count_children: число детей (0, если нет).
- start_date: дата заезда YYYY-MM-DD или null.
- nights: число ночей или null.
- price_per_day: цена за сутки (максимум из диапазона) или null (не 0).
- remarks: особые пожелания (море, кухня, удобства) или null.

Правила:
- "семья с одним ребенком" → 2 взрослых, 1 ребенок.
- "семья с двумя детьми" → 2 взрослых, 2 детей.
- "пара", "двое" → 2 взрослых, 0 детей.
- "один", "студент" → 1 взрослый, 0 детей.
- Если прямо написано "2 взрослых и 2 детей" — используй это.
- Даты: "30.06-7.07" → start_date = "2024-06-30", nights = 7 (разница).
- Цена: "1000-1500₽" → price_per_day = 1500.
- Не добавляй поля check_in_date, не пиши комментарии.

Схема:
{format_instructions}

Текст: {text}
""",
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = detailed_prompt | llm | parser

df = pd.read_csv('rental_26_annotated.csv', sep=';')
required_cols = ['text', 'count_adults', 'count_children', 'start_date', 'nights', 'price_per_day', 'remarks']
for col in required_cols:
    if col not in df.columns:
        raise KeyError(f"Нет столбца {col}")

results = []
for idx, row in df.iterrows():
    text = row['text']
    try:
        out = chain.invoke({"text": text})
        results.append(out)
        print(f"✅ {idx+1}: {out.get('count_adults')} взр, {out.get('count_children')} дет")
    except Exception as e:
        print(f"❌ {idx+1}: {e}")
        results.append(None)

df['model_output'] = results

fields = ['count_adults', 'count_children', 'start_date', 'nights', 'price_per_day', 'remarks']
acc = {}
for f in fields:
    correct = 0
    for i, row in df.iterrows():
        true_val = row[f]
        model_val = row['model_output'].get(f) if row['model_output'] else None
        if true_val == model_val or (pd.isna(true_val) and model_val is None):
            correct += 1
    acc[f] = correct / len(df)
    print(f"{f}: {acc[f]:.1%}")

avg = sum(acc.values()) / len(acc)
print(f"\nСредняя точность: {avg:.1%}")

df.to_csv("rental_26_structured_result.csv", index=False, encoding='utf-8-sig')
print("Сохранено в rental_26_structured_result.csv")

✅ 1: 2 взр, 0 дет
✅ 2: 3 взр, 3 дет
✅ 3: 2 взр, 0 дет
✅ 4: 3 взр, 0 дет
✅ 5: 2 взр, 2 дет
✅ 6: 3 взр, 0 дет
✅ 7: 2 взр, 0 дет
✅ 8: 2 взр, 1 дет
✅ 9: 2 взр, 1 дет
✅ 10: 2 взр, 3 дет
✅ 11: 2 взр, 0 дет
✅ 12: 2 взр, 2 дет
✅ 13: 2 взр, 2 дет
✅ 14: 1 взр, 0 дет
✅ 15: 2 взр, 2 дет
count_adults: 80.0%
count_children: 66.7%
start_date: 13.3%
nights: 6.7%
price_per_day: 80.0%
remarks: 13.3%

Средняя точность: 43.3%
Сохранено в rental_26_structured_result.csv
